In [ ]:
%matplotlib inline
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
import matplotlib.gridspec as gridspec
from matplotlib.colors import to_rgba
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 200,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
})

P = {
    'combined':   '#2563EB',
    'correct':    '#16A34A',
    'prm':        '#DC2626',
    'fmt':        '#7C3AED',
    'reward':     '#0891B2',
    'loss':       '#64748B',
    'batch_acc':  '#059669',
    'gt_match':   '#EA580C',
    'lccp':       '#0D9488',
    'step_acc':   '#7C3AED',
    'grounded':   '#16A34A',
    'sp':         '#2563EB',
    'q_reward':   '#DB2777',
    'lr':         '#92400E',
    'skip':       '#EF4444',
    'chain':      '#0369A1',
}

METRICS_PATH = Path('checkpoints/grpo/grpo_20260425_151304/metrics.jsonl')
rows = [json.loads(l) for l in METRICS_PATH.read_text().splitlines() if l.strip()]

def field(key, include_iter0=False):
    xs, ys = [], []
    for r in rows:
        if not include_iter0 and r['iteration'] == 0:
            continue
        v = r.get(key)
        if v is not None and v != '' and not (isinstance(v, float) and np.isnan(v)):
            try:
                xs.append(int(r['iteration']))
                ys.append(float(v))
            except (TypeError, ValueError):
                pass
    return np.array(xs), np.array(ys)

def field_all(key):
    return field(key, include_iter0=True)

train_rows = [r for r in rows if r['iteration'] > 0]
eval_rows  = [r for r in rows if r.get('combined_score') is not None]

RUN_NAME = METRICS_PATH.parent.name
N_ITERS  = max(r['iteration'] for r in rows)

# Phase bands for shading
phase_changes = []
prev = None
for r in train_rows:
    ph = r.get('training_phase', '')
    it = r['iteration']
    if ph != prev:
        phase_changes.append((it, ph))
        prev = ph

PHASE_COLORS = {
    'GROUNDED_ONLY': '#bbf7d0',
    'SELFPLAY_RAMP':  '#bfdbfe',
    'CONTINUOUS':     '#fde68a',
}

def shade_phases(ax, ymin=0, ymax=1.05):
    for i, (start, ph) in enumerate(phase_changes):
        end = phase_changes[i+1][0] if i+1 < len(phase_changes) else N_ITERS + 1
        color = PHASE_COLORS.get(ph, '#e2e8f0')
        ax.axvspan(start - 0.5, end - 0.5, alpha=0.25, color=color, zorder=0)
    handles = [mpatches.Patch(color=PHASE_COLORS.get(ph, '#e2e8f0'), alpha=0.5,
                              label=ph.replace('_', ' ').title())
               for _, ph in phase_changes]
    return handles

print(f'Run: {RUN_NAME}')
print(f'Total rows: {len(rows)}  |  Train iters: {len(train_rows)}  |  Eval iters: {len(eval_rows)}')
print(f'Phase transitions: {[(it, ph) for it, ph in phase_changes]}')

In [ ]:
# ── PLOT 1: Training Objective (combined_score) — Primary Demo Plot ───────────
xi_all, xv_all = field_all('combined_score')

fig, ax = plt.subplots(figsize=(11, 5))
ph_handles = shade_phases(ax)

ax.plot(xi_all, xv_all, color=P['combined'], linewidth=2.5,
        marker='o', markersize=6, label='Eval score', zorder=5)
ax.fill_between(xi_all, xv_all, alpha=0.12, color=P['combined'])

for x, y in zip(xi_all, xv_all):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=8.5, color=P['combined'], fontweight='bold')

if len(xv_all) >= 2:
    delta = xv_all[-1] - xv_all[0]
    sign = '+' if delta >= 0 else ''
    ax.set_title(
        f'GRPO Training — Combined Reward Score   '
        f'(baseline→final Δ = {sign}{delta:+.3f} = {sign}{delta*100:.1f} pp)',
        fontsize=12, fontweight='bold'
    )

ax.set_xlabel('Iteration')
ax.set_ylabel('Score')
ax.set_ylim(0.8, 1.02)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
ax.set_xticks(range(0, N_ITERS + 1, 5))
ax.legend(handles=[ax.lines[0]] + ph_handles, loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:
# ── PLOT 2: Correctness — gt_match_rate (every iter) + correct_rate (eval iters) ──
gi, gv = field('gt_match_rate')
ci, cv = field_all('correct_rate')

fig, ax = plt.subplots(figsize=(11, 5))
ph_handles = shade_phases(ax)

if len(gv):
    ax.plot(gi, gv, color=P['gt_match'], linewidth=1.6, alpha=0.85,
            marker='.', markersize=5, label='gt_match_rate (per training iter)')
    ax.fill_between(gi, gv, alpha=0.07, color=P['gt_match'])

if len(cv):
    ax.plot(ci, cv, color=P['correct'], linewidth=2.5,
            marker='D', markersize=7, label='correct_rate (eval, held-out)', zorder=5)
    for x, y in zip(ci, cv):
        ax.annotate(f'{y:.0%}', (x, y), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=8.5, color=P['correct'], fontweight='bold')

ax.set_xlabel('Iteration')
ax.set_ylabel('Correctness Rate')
ax.set_ylim(0.5, 1.02)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xticks(range(0, N_ITERS + 1, 2))
ax.set_title('Answer Correctness — Training Batch (gt_match) vs Held-Out Eval', fontweight='bold')
ax.legend(handles=ax.lines[:2] + ph_handles, loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:
# ── PLOT 3: Process Quality — PRM, Step Accuracy, LCCP ───────────────────────
prm_ti, prm_tv = field('prm_mean')        # eval only
step_i, step_v = field('step_accuracy')   # every iter
lccp_i, lccp_v = field('lccp')            # every iter
prm_ei, prm_ev = field_all('prm_mean')    # includes iter 0

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (xi, xv, color, label, title) in zip(axes, [
    (prm_ei, prm_ev,  P['prm'],      'PRM mean (eval)', 'PRM Step Quality'),
    (step_i, step_v,  P['step_acc'], 'step accuracy',   'Step Accuracy (per iter)'),
    (lccp_i, lccp_v,  P['lccp'],     'LCCP',            'Chain Integrity (LCCP)'),
]):
    shade_phases(ax)
    ax.plot(xi, xv, color=color, linewidth=2, marker='o', markersize=4, label=label)
    ax.fill_between(xi, xv, alpha=0.1, color=color)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    if len(xv):
        ax.annotate(f'{xv[-1]:.2%}', (xi[-1], xv[-1]), textcoords='offset points',
                    xytext=(6, 4), fontsize=9, color=color, fontweight='bold')

fig.suptitle('Process Quality Metrics', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# ── PLOT 4: Training Dynamics — Loss, Mean Reward, Batch Accuracy ────────────
loss_i, loss_v   = field('loss')
rew_i,  rew_v    = field('mean_reward')
std_i,  std_v    = field('std_reward')
bacc_i, bacc_v   = field('batch_accuracy')
gacc_i, gacc_v   = field('grounded_accuracy')

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

shade_phases(axes[0])
axes[0].plot(loss_i, loss_v, color=P['loss'], linewidth=1.8, marker='.', markersize=4)
axes[0].fill_between(loss_i, loss_v, alpha=0.12, color=P['loss'])
axes[0].set_ylabel('GRPO Loss')
axes[0].set_title('Training Loss', fontweight='bold')

shade_phases(axes[1])
if len(rew_v) and len(std_v):
    axes[1].plot(rew_i, rew_v, color=P['reward'], linewidth=1.8, marker='.', markersize=4, label='mean reward')
    axes[1].fill_between(rew_i,
                         np.array(rew_v) - np.array(std_v),
                         np.array(rew_v) + np.array(std_v),
                         alpha=0.12, color=P['reward'], label='±1 std')
axes[1].set_ylabel('Reward')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Mean Batch Reward ± Std', fontweight='bold')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
axes[1].legend(loc='lower right')

shade_phases(axes[2])
axes[2].plot(bacc_i, bacc_v, color=P['batch_acc'], linewidth=1.8, marker='.', markersize=4, label='batch acc (all)')
axes[2].plot(gacc_i, gacc_v, color=P['grounded'], linewidth=1.5, linestyle='--', marker='.', markersize=4, label='grounded acc')
axes[2].fill_between(bacc_i, bacc_v, alpha=0.08, color=P['batch_acc'])
axes[2].set_ylabel('Accuracy')
axes[2].set_ylim(0, 1.05)
axes[2].set_title('Batch Accuracy', fontweight='bold')
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
axes[2].legend(loc='lower right')
axes[2].set_xlabel('Iteration')

fig.suptitle('GRPO Training Dynamics', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# ── PLOT 5: Phase Curriculum & Self-Play Ramp ─────────────────────────────────
sp_i,  sp_v  = field('effective_sp_ratio')
nsp_i, nsp_v = field('n_self_play_groups')
ng_i,  ng_v  = field('n_groups')
sk_i,  sk_v  = field('skipped_groups')

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

shade_phases(axes[0])
axes[0].plot(sp_i, sp_v * 100, color=P['sp'], linewidth=2.2, marker='o', markersize=5, label='self-play ratio')
axes[0].fill_between(sp_i, sp_v * 100, alpha=0.12, color=P['sp'])
axes[0].axhline(70, color=P['sp'], linewidth=1, linestyle=':', alpha=0.5, label='target ceiling (70%)')
axes[0].set_ylabel('Self-Play Ratio (%)')
axes[0].set_ylim(-2, 80)
axes[0].set_title('Self-Play Ratio Ramp (Phase 1 → Phase 2 → Continuous)', fontweight='bold')
axes[0].legend(loc='upper left')

shade_phases(axes[1])
axes[1].bar(ng_i, ng_v, color=P['batch_acc'], alpha=0.75, label='valid groups')
axes[1].bar(sk_i, sk_v, bottom=0, color=P['skip'], alpha=0.65, label='skipped groups')
axes[1].set_ylabel('Groups per Iteration')
axes[1].set_title('Valid vs Skipped GRPO Groups per Iteration', fontweight='bold')
axes[1].set_xlabel('Iteration')
axes[1].legend(loc='upper left')

fig.suptitle('Curriculum Phase & Group Statistics', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# ── PLOT 6: Learning Rate Schedule ───────────────────────────────────────────
lr_i, lr_v = field('learning_rate')

fig, ax = plt.subplots(figsize=(10, 4))
shade_phases(ax)
ax.plot(lr_i, lr_v, color=P['lr'], linewidth=2, marker='o', markersize=4)
ax.fill_between(lr_i, lr_v, alpha=0.12, color=P['lr'])
ax.set_yscale('log')
ax.set_xlabel('Iteration')
ax.set_ylabel('Learning Rate (log scale)')
ax.set_title('Learning Rate Schedule: Linear Warmup → Cosine Decay', fontweight='bold')
if len(lr_v):
    ax.annotate(f'peak\n{max(lr_v):.2e}', (lr_i[np.argmax(lr_v)], max(lr_v)),
                textcoords='offset points', xytext=(6, 8), fontsize=9, color=P['lr'])
    ax.annotate(f'final\n{lr_v[-1]:.2e}', (lr_i[-1], lr_v[-1]),
                textcoords='offset points', xytext=(-45, 8), fontsize=9, color=P['lr'])
fig.tight_layout()
plt.show()

In [ ]:
# ── PLOT 7: Self-Play Question Generation Quality ─────────────────────────────
qr_i,  qr_v  = field('mean_question_reward')
qq_i,  qq_v  = field('q_quality_rate')
qt_i,  qt_v  = field('q_topic_match')
qd_i,  qd_v  = field('q_difficulty_fit')
qc_i,  qc_v  = field('q_clarity')
qn_i,  qn_v  = field('q_novelty')
qs_i,  qs_v  = field('q_solvability')

has_sp = len(qr_v) > 0
if not has_sp:
    print('No self-play groups yet — this plot will appear once self-play begins (iter 8+).')
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    specs = [
        (qr_i, qr_v,  P['q_reward'],   'Mean Question Reward'),
        (qq_i, qq_v,  P['batch_acc'],  'Question Quality Rate (>0.5)'),
        (qt_i, qt_v,  P['combined'],   'Topic Match'),
        (qd_i, qd_v,  P['gt_match'],   'Difficulty Fit'),
        (qc_i, qc_v,  P['prm'],        'Clarity'),
        (qn_i, qn_v,  P['lccp'],       'Novelty'),
    ]

    for ax, (xi, xv, color, title) in zip(axes, specs):
        shade_phases(ax)
        if len(xi):
            ax.plot(xi, xv, color=color, linewidth=2, marker='o', markersize=4)
            ax.fill_between(xi, xv, alpha=0.1, color=color)
            ax.set_ylim(0, 1.05)
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
            if len(xv):
                ax.annotate(f'{xv[-1]:.2%}', (xi[-1], xv[-1]), textcoords='offset points',
                            xytext=(6, 4), fontsize=9, color=color, fontweight='bold')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Iteration')

    fig.suptitle('Self-Play: Question Generation Quality Breakdown', fontsize=13, fontweight='bold')
    fig.tight_layout()
    plt.show()

In [ ]:
# ── PLOT 8: Eval Score Breakdown — Stacked Area (weighted components) ─────────
ei, ev = field_all('combined_score')
if len(ei) >= 2:
    it_map = {r['iteration']: r for r in rows}
    iters = sorted(ei)

    weights = {'correct': 0.50, 'prm': 0.15, 'fmt': 0.10}
    keys    = {'correct': 'correct_rate', 'prm': 'prm_mean', 'fmt': 'format_mean'}
    aligned = {k: [] for k in weights}

    for it in iters:
        r = it_map.get(it, {})
        for comp, field_key in keys.items():
            v = r.get(field_key)
            aligned[comp].append(float(v) * weights[comp] if (v is not None and v != '') else 0.0)

    x   = np.array(iters)
    arr = np.array([aligned['correct'], aligned['prm'], aligned['fmt']])

    fig, ax = plt.subplots(figsize=(11, 5))
    labels = ['Correct (×0.50)', 'PRM (×0.15)', 'Format (×0.10)']
    colors = [P['correct'], P['prm'], P['fmt']]
    ax.stackplot(x, arr, labels=labels, colors=colors, alpha=0.78)
    ax.plot(x, ev, color='black', linewidth=2, linestyle='--', label='Combined score', zorder=5, marker='D', markersize=5)

    for xi, yi in zip(x, ev):
        ax.annotate(f'{yi:.3f}', (xi, yi), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=8, color='black')

    ax.set_xlabel('Iteration')
    ax.set_ylabel('Weighted Score Contribution')
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.set_title('Eval Score Decomposition — Stacked Weighted Components', fontweight='bold')
    ax.legend(loc='lower right', ncol=2)
    fig.tight_layout()
    plt.show()

In [ ]:
# ── PLOT 9: Training Reward vs Eval Score (dual overlay) ──────────────────────
ri, rv = field('mean_reward')
ei, ev = field_all('combined_score')

fig, ax = plt.subplots(figsize=(11, 5))
ph_handles = shade_phases(ax)

ax.plot(ri, rv, color=P['reward'], linewidth=1.4, alpha=0.7,
        marker='.', markersize=4, label='Batch reward (training rollouts)')
ax.fill_between(ri, rv, alpha=0.06, color=P['reward'])

ax.plot(ei, ev, color=P['combined'], linewidth=2.8, marker='D', markersize=7,
        label='Eval score (held-out)', zorder=6)
for x, y in zip(ei, ev):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9,
                color=P['combined'], fontweight='bold')

ax.set_xlabel('Iteration')
ax.set_ylabel('Score')
ax.set_ylim(0.7, 1.02)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
ax.set_title('Training Batch Reward vs Held-Out Eval Score', fontweight='bold')
ax.legend(handles=[ax.lines[0], ax.lines[1]] + ph_handles, loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:
# ── PLOT 10: Chain Scoring Calibration (Phase 2 shadow mode) ──────────────────
corr_i, corr_v  = field('chain_prm_correlation')
succ_i, succ_v  = field('extraction_success_rate')
ci_i,   ci_v    = field('chain_integrity_score')
spc_i,  spc_v   = field('sp_chain_integrity_score')
active_i, active_v = field('chain_scoring_active')

if any(v > 0 for v in corr_v) if len(corr_v) else False:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.flatten()

    specs = [
        (corr_i,   corr_v,  P['chain'],    'Chain↔PRM Pearson Correlation',   'Correlation',  False),
        (succ_i,   succ_v,  P['correct'],  'Extraction Success Rate',          'Rate',         True),
        (ci_i,     ci_v,    P['chain'],    'Chain Integrity Score (grounded)', 'Score',        True),
        (spc_i,    spc_v,   P['sp'],       'SP Chain Integrity Score',         'Score',        True),
    ]
    for ax, (xi, xv, color, title, ylabel, pct) in zip(axes, specs):
        shade_phases(ax)
        if len(xi):
            ax.plot(xi, xv, color=color, linewidth=2, marker='o', markersize=4)
            ax.fill_between(xi, xv, alpha=0.1, color=color)
            if pct:
                ax.set_ylim(0, 1.05)
                ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Iteration')
        ax.set_ylabel(ylabel)

    # shade when chain scoring is active
    for ax in axes:
        for i, (it, av) in enumerate(zip(active_i, active_v)):
            if av > 0:
                ax.axvspan(it - 0.5, it + 0.5, color='gold', alpha=0.3, zorder=0)

    fig.suptitle('Chain Scoring Calibration (Phase 2 Shadow Mode)', fontsize=13, fontweight='bold')
    fig.tight_layout()
    plt.show()
else:
    print('Chain scoring calibration window not yet reached in this run (needs SELFPLAY_RAMP + 50 shadow samples).')

In [ ]:
# ── PLOT 11: Full Summary Dashboard ───────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f'GRPO Training Summary Dashboard\n{RUN_NAME}', fontsize=15, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

panel_specs = [
    (gs[0, :2],  'combined_score',   P['combined'],  'Eval: Combined Score',       True,  True),
    (gs[0, 2:],  'gt_match_rate',    P['gt_match'],  'Train: GT Match Rate',       False, True),
    (gs[1, 0],   'correct_rate',     P['correct'],   'Eval: Correctness',          True,  True),
    (gs[1, 1],   'prm_mean',         P['prm'],       'Eval: PRM Step Quality',     True,  True),
    (gs[1, 2],   'step_accuracy',    P['step_acc'],  'Train: Step Accuracy',       False, True),
    (gs[1, 3],   'lccp',             P['lccp'],      'Train: LCCP',                False, True),
    (gs[2, 0],   'loss',             P['loss'],      'Train: GRPO Loss',           False, False),
    (gs[2, 1],   'mean_reward',      P['reward'],    'Train: Mean Reward',         False, True),
    (gs[2, 2],   'learning_rate',    P['lr'],        'Train: Learning Rate',       False, False),
    (gs[2, 3],   'effective_sp_ratio', P['sp'],      'Train: SP Ratio',            False, True),
]

for spec, field_key, color, title, eval_only, pct in panel_specs:
    ax = fig.add_subplot(spec)
    xi, xv = (field_all if eval_only else field)(field_key)
    shade_phases(ax)
    if len(xi):
        ax.plot(xi, xv, color=color, linewidth=1.8, marker='o', markersize=3)
        ax.fill_between(xi, xv, alpha=0.1, color=color)
        if pct:
            ax.set_ylim(0, 1.05)
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
        if field_key == 'learning_rate':
            ax.set_yscale('log')
        if len(xv):
            ax.annotate(f'{xv[-1]:.3f}', (xi[-1], xv[-1]), textcoords='offset points',
                        xytext=(5, 3), fontsize=8, color=color, fontweight='bold')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Iter', fontsize=8)

plt.savefig('checkpoints/grpo/grpo_20260425_151304/summary_dashboard.png',
            dpi=200, bbox_inches='tight')
plt.show()
print('Dashboard saved to checkpoints/grpo/grpo_20260425_151304/summary_dashboard.png')

In [ ]:
# ── PLOT 12: Numeric Summary Table ────────────────────────────────────────────
from IPython.display import display, HTML

iter0  = next((r for r in rows if r['iteration'] == 0), {})
last_eval = next((r for r in reversed(rows) if r.get('combined_score') is not None), {})
best_eval = max((r for r in rows if r.get('combined_score') is not None),
                key=lambda r: r['combined_score'], default={})

metrics = [
    ('Combined Score',    'combined_score', '{:.4f}'),
    ('Correct Rate',      'correct_rate',   '{:.1%}'),
    ('PRM Mean',          'prm_mean',       '{:.4f}'),
    ('Step Accuracy',     'step_accuracy',  '{:.1%}'),
    ('LCCP',              'lccp',           '{:.1%}'),
    ('Format Mean',       'format_mean',    '{:.3f}'),
]

def fmt_val(r, key, fmt):
    v = r.get(key)
    return fmt.format(v) if v is not None and v != '' else '—'

rows_html = ''
for label, key, fmt in metrics:
    b0  = fmt_val(iter0,     key, fmt)
    bl  = fmt_val(last_eval, key, fmt)
    bb  = fmt_val(best_eval, key, fmt)
    try:
        v0 = iter0.get(key)
        vl = last_eval.get(key)
        delta = float(vl) - float(v0) if (v0 is not None and vl is not None) else None
        sign = '+' if delta and delta >= 0 else ''
        delta_str = f'{sign}{delta*100:.2f} pp' if delta is not None else '—'
        delta_color = '#16A34A' if (delta is not None and delta >= 0) else '#DC2626'
    except:
        delta_str = '—'
        delta_color = '#374151'
    rows_html += (
        f'<tr><td style="font-weight:600">{label}</td>'
        f'<td>{b0}</td><td>{bl}</td><td>{bb}</td>'
        f'<td style="color:{delta_color};font-weight:600">{delta_str}</td></tr>'
    )

html = f'''
<style>
  .grpo-table {{ border-collapse: collapse; width: 680px; font-family: monospace; font-size: 13px; }}
  .grpo-table th {{ background: #1e3a5f; color: white; padding: 8px 14px; text-align: left; }}
  .grpo-table td {{ padding: 7px 14px; border-bottom: 1px solid #e5e7eb; }}
  .grpo-table tr:hover {{ background: #f0f9ff; }}
</style>
<h3 style="font-family:sans-serif">Run: <code>{RUN_NAME}</code> &nbsp;·&nbsp; {N_ITERS} iterations</h3>
<table class="grpo-table">
  <tr>
    <th>Metric</th>
    <th>Baseline (iter 0)</th>
    <th>Final (iter {last_eval.get("iteration","?")})</th>
    <th>Best (iter {best_eval.get("iteration","?")})</th>
    <th>Δ (baseline→final)</th>
  </tr>
  {rows_html}
</table>
'''
display(HTML(html))